In [1]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

In [2]:
# Define the data model
def create_data_model():
    """Stores the data for the problem."""
    data = {
        'time_matrix': [
            [0, 10, 20, 30],
            [10, 0, 25, 35],
            [20, 25, 0, 15],
            [30, 35, 15, 0],
        ],
        'service_times': [2, 3, 3, 1],
        'num_vehicles': 3,
         # Demand at each location (index 0 is the depot)
        'demands':[0, 15, 10, 3],
        # Vehicle capacities
        'vehicle_capacities': [20, 20, 40],
        'vehicle_types': ['EV', 'EV', 'ICE'],
        'vehicle_factors':[1, 1, 10], # EV to ICE multiplied to the service times so that EVs have a shorter service time
        'depot': 0,
    }
    return data

In [3]:
# Define the callback for vehicle-dependent travel times
def create_time_callback(data, vehicle_index):
    def time_callback(from_index, to_index):
        # Convert indices to nodes
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        total_time = data['time_matrix'][from_node][to_node] + (data['service_times'][from_node] * data['vehicle_factors'][vehicle_index])
        return total_time
    return time_callback

In [4]:
# Print the solution
def print_solution(data, manager, routing, solution):
    total_time = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_time = 0
        route = []
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_time += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
        route.append(manager.IndexToNode(index))
        total_time += route_time
        print(f"Route for vehicle {vehicle_id}: {route}")
        print(f"Travel time for vehicle {vehicle_id}: {route_time}")
    print(f"Total travel time: {total_time}")

In [5]:
# Instantiate the data problem
data = create_data_model()
manager = pywrapcp.RoutingIndexManager(len(data['time_matrix']), data['num_vehicles'], data['depot'])
routing = pywrapcp.RoutingModel(manager)

# Create and register callbacks for each vehicle
transit_callback_indices = []
time_dim = 'Time'

for vehicle_id in range(data['num_vehicles']):
    callback = create_time_callback(data, vehicle_id)
    transit_callback_index = routing.RegisterTransitCallback(callback)
    
    transit_callback_indices.append(transit_callback_index)
    routing.SetArcCostEvaluatorOfVehicle(transit_callback_index, vehicle_id)

    routing.AddDimension(
        transit_callback_index,
        30,  # allow waiting time
        86400,  # maximum time per vehicle, JU: set to minutes in a day assuming no trip goes beyod a day
        True,  # Don't force start cumul to zero.
        time_dim)

time_dimension = routing.GetDimensionOrDie(time_dim)

#print('transits: ', transit_callback_indices)

# Add capacity constraints
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  # no slack
    data['vehicle_capacities'],  # vehicle maximum capacities
    True,  # start cumul at zero
    'Capacity')

# Define search parameters
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

# Solve the problem
solution = routing.SolveWithParameters(search_parameters)



In [6]:
# Print the solution
if solution:
    print_solution(data, manager, routing, solution)
else:
    print("No solution found!")

Route for vehicle 0: [0, 2, 3, 0]
Travel time for vehicle 0: 71
Route for vehicle 1: [0, 1, 0]
Travel time for vehicle 1: 25
Route for vehicle 2: [0, 0]
Travel time for vehicle 2: 0
Total travel time: 96
